# 实验4.4 网络知识蒸馏（Knowledge Distillation）实验教程

> **运行环境**：cann_9.0.0-py3.11-A2-arm-20260715 | ASCEND, 1*NPU 910B3, 16vCPUs, 32GiB
> **适用对象**：深度学习初学者
> **学习方式**：从上到下逐格运行，边学边练

---

## 🎯 学习目标

通过本 Notebook 你将学会：

1. 理解 **知识蒸馏（Knowledge Distillation, KD）** 的基本原理与作用
2. 搭建 **教师网络（Teacher）** 与 **学生网络（Student）**
3. 理解 **温度软化（Temperature Softmax）** 与 **暗知识（Dark Knowledge）**
4. 实现 **蒸馏损失函数** = 硬标签损失 + 软标签损失
5. 训练学生网络：**有蒸馏 vs 无蒸馏**，对比精度与参数量
6. 在昇腾 910 NPU 上运行训练，体会蒸馏的 "用小模型逼近大模型" 思想

## 📖 实验流程总览

```
准备数据 → 定义教师网络(大) → 定义学生网络(小)
  → 训练教师网络 → 评估教师
  → 蒸馏训练学生(用教师的软标签) → 普通训练学生(只用硬标签)
  → 三者对比(精度/参数量/推理速度) → 可视化软标签 → 总结
```

> 💡 **提示**：请按顺序逐个运行代码格（Cell），观察每一步的输出，配合 Markdown 说明理解每一步在做什么。

## 一、什么是知识蒸馏（Knowledge Distillation）？

### 1.1 为什么需要蒸馏？

在深度学习中，**大模型精度高但慢**，**小模型快但精度低**。
知识蒸馏的目标是：

> 让一个 **小模型（学生）** 学习一个 **大模型（教师）** 的知识，
> 从而在参数量大幅减少的同时，尽可能保持接近教师的精度。

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">角色</th>
<th style="text-align: left;">说明</th>
</tr>
<tr>
<td style="text-align: left;">教师 Teacher</td>
<td style="text-align: left;">大模型，精度高，已训练好，推理慢</td>
</tr>
<tr>
<td style="text-align: left;">学生 Student</td>
<td style="text-align: left;">小模型，参数少，推理快，待训练</td>
</tr>
<tr>
<td style="text-align: left;">蒸馏 Distillation</td>
<td style="text-align: left;">把教师的知识"提炼"给学生的过程</td>
</tr>
</table>

**角色关系详解**：
- **教师网络**：通常是一个参数量大、层数深的大模型，已经在任务上训练到较高精度。教师的输出不仅包含"正确答案"（概率最大的类别），还包含各类别之间的相对关系（即"暗知识"）。教师只在蒸馏过程中做推理，不更新参数。
- **学生网络**：参数量小、层数浅的小模型，推理速度快、内存占用少。学生是蒸馏的训练对象，通过学习教师的输出分布来提升精度。
- **蒸馏过程**：学生同时学习两类信息——**硬标签**（真实类别，如"这是数字 3"）和**软标签**（教师的概率分布，如"最可能是 3，其次像 5"）。软标签提供的类间关系信息比硬标签更丰富，是蒸馏提升学生精度的关键。

### 1.2 蒸馏的核心思想

传统训练只用 **硬标签（Hard Label）**——即真实的 one-hot 类别。
但硬标签只告诉学生"这是 3"，没告诉学生"这看起来有点像 5 和 8"。

教师的输出概率分布包含了更丰富的信息，例如：

```
教师输出: [0.01, 0.01, 0.02, 0.85, 0.01, 0.05, 0.01, 0.01, 0.02, 0.01]
           类0   类1   类2   类3   类4   类5   类6   类7   类8   类9
```

这个分布不仅说"最可能是 3"，还说"其次有点像 5"——这就是 **暗知识（Dark Knowledge）**。
学生学这个分布，比只学 one-hot 能获得更多 "类间关系" 信息。

### 1.3 温度软化（Temperature Softmax）

标准 softmax 会把 logits 放大，使概率分布很 "尖锐"（一个类接近 1，其余接近 0），
暗知识被淹没。引入 **温度 T** 让分布变 "平滑"，暴露类间关系：

$$p_i = \frac{\exp(z_i / T)}{\sum_j \exp(z_j / T)}$$

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">温度 T</th>
<th style="text-align: left;">效果</th>
</tr>
<tr>
<td style="text-align: left;">T = 1</td>
<td style="text-align: left;">标准 softmax，分布尖锐</td>
</tr>
<tr>
<td style="text-align: left;">T > 1</td>
<td style="text-align: left;">分布平滑，暴露暗知识（常用 2~10）</td>
</tr>
<tr>
<td style="text-align: left;">T → ∞</td>
<td style="text-align: left;">分布趋于均匀</td>
</tr>
</table>

**温度软化原理详解**：
- 标准 softmax（T=1）会将 logits 指数放大，使最大类别的概率接近 1，其余接近 0。这种"尖锐"分布几乎只包含"正确答案"信息，暗知识被淹没。
- 引入温度 T>1 后，logits 除以 T 再做 softmax，相当于"压缩"logits 的差距，使概率分布变"平滑"。非 top 类的概率提高，暴露了类间的相对关系。
- 例如 logits=[2.0, 1.0, 0.5]：T=1 时 softmax≈[0.58, 0.21, 0.13]；T=4 时 softmax≈[0.39, 0.31, 0.28]，分布更均匀，类间关系更明显。
- T 太大时分布趋于均匀（各类概率相等），失去判别力；T 太小时分布过于尖锐，暗知识不明显。常用 T=2~10。

### 1.4 蒸馏损失函数

学生网络的总损失由两部分加权组成：

$$L = \alpha \cdot L_{hard} + (1 - \alpha) \cdot T^2 \cdot L_{soft}$$

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">项</th>
<th style="text-align: left;">含义</th>
</tr>
<tr>
<td style="text-align: left;">$L_{hard}$</td>
<td style="text-align: left;">学生输出与真实标签的交叉熵（硬标签损失）</td>
</tr>
<tr>
<td style="text-align: left;">$L_{soft}$</td>
<td style="text-align: left;">学生软分布与教师软分布的 KL 散度（软标签损失）</td>
</tr>
<tr>
<td style="text-align: left;">$T^2$</td>
<td style="text-align: left;">补偿温度对梯度的缩放，使软损失梯度量级与硬损失匹配</td>
</tr>
<tr>
<td style="text-align: left;">$\alpha$</td>
<td style="text-align: left;">硬损失权重，常用 0.5</td>
</tr>
</table>

**损失函数各项详解**：
- **$L_{hard}$（硬标签损失）**：学生输出与真实标签的交叉熵，和普通训练完全一样。它确保学生学到正确的分类能力，是"保底"项。
- **$L_{soft}$（软标签损失）**：学生软分布与教师软分布的 KL 散度。它让学生模仿教师的输出分布（包括暗知识），是蒸馏的"知识迁移"项。
- **$T^2$ 补偿因子**：温度 softmax 使梯度缩小为原来的 1/T²（因为 logits 除以 T 后对 logits 的导数为 1/T）。乘以 T² 将梯度量级恢复，使软损失与硬损失的梯度量级匹配，避免软损失被"淹没"。
- **$\alpha$ 权重**：控制硬损失和软损失的比重。$\alpha=0.5$ 表示两者各占一半；$\alpha$ 越大越依赖真实标签，越小越依赖教师分布。

> 📖 **论文**：Hinton et al. *Distilling the Knowledge in a Neural Network* (2015)

## 二、环境准备

导入所需库，并检测运行设备。

- **昇腾 910 NPU**：通过 `torch_npu` 插件使用，训练阶段加速
- **CPU 回退**：没有 NPU/GPU 时自动使用 CPU

> ⚠️ 在 GitCode 昇腾 Notebook 中，`torch` 与 `torch_npu` 通常已预装。

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import os
import time
import copy

print('PyTorch 版本:', torch.__version__)

# ---- 设备检测：优先昇腾NPU，其次CUDA，最后CPU ----
try:
    import torch_npu  # 昇腾NPU插件
    device = torch.device('npu')
    print('✅ 检测到昇腾 NPU，训练将使用:', device)
except ImportError:
    if torch.cuda.is_available():
        device = torch.device('cuda')
        print('✅ 检测到 CUDA GPU，训练将使用:', device)
    else:
        device = torch.device('cpu')
        print('ℹ️  未检测到NPU/GPU，训练将使用 CPU')

print('当前设备:', device)

## 三、准备数据集

本实验用 **随机合成数据** 模拟 MNIST 风格的手写数字识别任务：

- 输入：`1×28×28` 灰度图
- 输出：10 类（数字 0~9）

> 为了让对比更公平、可复现，我们 **固定随机种子** 生成训练集和测试集。
> 实际项目中可替换为 `torchvision.datasets.MNIST` 的 DataLoader。

In [ ]:
torch.manual_seed(42)

NUM_TRAIN = 800   # 训练样本数
NUM_TEST  = 200   # 测试样本数

# ---- 生成训练集 ----
train_images = torch.randn(NUM_TRAIN, 1, 28, 28)
train_labels = torch.randint(0, 10, (NUM_TRAIN,))

# ---- 生成测试集 ----
test_images = torch.randn(NUM_TEST, 1, 28, 28)
test_labels = torch.randint(0, 10, (NUM_TEST,))

print(f'训练集: {train_images.shape}, 标签: {train_labels.shape}')
print(f'测试集: {test_images.shape}, 标签: {test_labels.shape}')
print(f'标签范围: {train_labels.min().item()} ~ {train_labels.max().item()}')

# 移到设备
train_images = train_images.to(device)
train_labels = train_labels.to(device)
test_images  = test_images.to(device)
test_labels  = test_labels.to(device)

## 四、定义教师网络（Teacher）

教师是一个 **较大的 CNN**，参数多、容量大、精度高：

```
输入(1×28×28)
  → Conv2d(1→64, 3×3) → ReLU
  → Conv2d(64→128, 3×3) → ReLU
  → Flatten → Linear(128*28*28 → 256) → ReLU
  → Linear(256 → 10)
输出(10类 logits)
```

> 教师网络只输出 **logits**（未经 softmax 的原始分数），
> 蒸馏时再手动做温度 softmax。

In [ ]:
class TeacherNet(nn.Module):
    def __init__(self):
        super(TeacherNet, self).__init__()
        self.conv1 = nn.Conv2d(1, 64, 3, 1, 1)    # 1→64通道
        self.relu1 = nn.ReLU()
        self.conv2 = nn.Conv2d(64, 128, 3, 1, 1)  # 64→128通道
        self.relu2 = nn.ReLU()
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(128 * 28 * 28, 256)
        self.relu3 = nn.ReLU()
        self.fc2 = nn.Linear(256, 10)

    def forward(self, x):
        x = self.relu1(self.conv1(x))
        x = self.relu2(self.conv2(x))
        x = self.flatten(x)
        x = self.relu3(self.fc1(x))
        x = self.fc2(x)
        return x  # 返回logits

teacher = TeacherNet().to(device)
teacher_params = sum(p.numel() for p in teacher.parameters())
print('===== 教师网络结构 =====')
print(teacher)
print(f'\n教师网络参数量: {teacher_params:,}')

## 五、定义学生网络（Student）

学生是一个 **较小的 CNN**，参数少、推理快：

```
输入(1×28×28)
  → Conv2d(1→16, 3×3) → ReLU
  → Conv2d(16→32, 3×3) → ReLU
  → Flatten → Linear(32*28*28 → 10)
输出(10类 logits)
```

对比教师与学生：

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;"></th>
<th style="text-align: left;">教师</th>
<th style="text-align: left;">学生</th>
</tr>
<tr>
<td style="text-align: left;">Conv 通道</td>
<td style="text-align: left;">64→128</td>
<td style="text-align: left;">16→32</td>
</tr>
<tr>
<td style="text-align: left;">隐藏层</td>
<td style="text-align: left;">有(256)</td>
<td style="text-align: left;">无</td>
</tr>
<tr>
<td style="text-align: left;">参数量</td>
<td style="text-align: left;">多</td>
<td style="text-align: left;">少</td>
</tr>
</table>

> 学生网络比教师小很多，这正是蒸馏的意义——小模型学大模型的知识。

**教师与学生架构对比详解**：
- **教师网络**：Conv2d(1→64) → Conv2d(64→128) → Linear(128×28×28→256) → Linear(256→10)。两个卷积层通道数较大（64、128），且有两个全连接层（中间有 256 维隐藏层），参数量约 12.8M。
- **学生网络**：Conv2d(1→16) → Conv2d(16→32) → Linear(32×28×28→10)。卷积通道数小（16、32），只有一个全连接层（无隐藏层），参数量约 0.8M。
- **压缩比**：教师/学生 ≈ 16x，即学生参数量仅为教师的约 6%。蒸馏的目标是让这个 6% 大小的学生达到尽可能接近教师的精度。


In [ ]:
class StudentNet(nn.Module):
    def __init__(self):
        super(StudentNet, self).__init__()
        self.conv1 = nn.Conv2d(1, 16, 3, 1, 1)    # 1→16通道
        self.relu1 = nn.ReLU()
        self.conv2 = nn.Conv2d(16, 32, 3, 1, 1)   # 16→32通道
        self.relu2 = nn.ReLU()
        self.flatten = nn.Flatten()
        self.fc = nn.Linear(32 * 28 * 28, 10)

    def forward(self, x):
        x = self.relu1(self.conv1(x))
        x = self.relu2(self.conv2(x))
        x = self.flatten(x)
        x = self.fc(x)
        return x  # 返回logits

student = StudentNet().to(device)
student_params = sum(p.numel() for p in student.parameters())
print('===== 学生网络结构 =====')
print(student)
print(f'\n学生网络参数量: {student_params:,}')
print(f'\n参数压缩比: 教师/学生 = {teacher_params/student_params:.2f}x')
print(f'学生参数量仅为教师的 {100*student_params/teacher_params:.2f}%')

## 六、训练与评估工具函数

先定义通用的训练和评估函数，后面教师、学生都会用到。

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">函数</th>
<th style="text-align: left;">作用</th>
</tr>
<tr>
<td style="text-align: left;"><code>evaluate(model, ...)</code></td>
<td style="text-align: left;">在测试集上计算准确率</td>
</tr>
<tr>
<td style="text-align: left;"><code>measure_time(model)</code></td>
<td style="text-align: left;">测 200 次推理总耗时</td>
</tr>
<tr>
<td style="text-align: left;"><code>get_model_size(model)</code></td>
<td style="text-align: left;">获取模型权重文件大小(KB)</td>
</tr>
</table>

In [ ]:
def evaluate(model, images, labels):
    """在测试集上评估准确率"""
    model.eval()
    correct = 0
    with torch.no_grad():
        # 分batch评估，避免一次性显存占用过大
        batch_size = 50
        for i in range(0, images.size(0), batch_size):
            x = images[i:i+batch_size]
            y = labels[i:i+batch_size]
            pred = model(x).argmax(dim=1)
            correct += (pred == y).sum().item()
    return correct / images.size(0)


def measure_time(model, device):
    """测量200次推理总耗时(秒)"""
    model.eval()
    x = torch.randn(1, 1, 28, 28).to(device)
    with torch.no_grad():
        for _ in range(10):  # 预热
            model(x)
    if device.type == 'npu':
        torch.npu.synchronize()
    start = time.time()
    with torch.no_grad():
        for _ in range(200):
            model(x)
    if device.type == 'npu':
        torch.npu.synchronize()
    end = time.time()
    return end - start


def get_model_size(model):
    """返回模型state_dict文件大小(KB)"""
    torch.save(model.state_dict(), 'temp.p')
    size = os.path.getsize('temp.p') / 1024
    os.remove('temp.p')
    return size

print('✅ 工具函数定义完成')

## 七、训练教师网络

用标准交叉熵损失训练教师网络，这是蒸馏的 **第一步**——
教师必须先训练好，才能指导学生。

> 训练时使用小 batch 随机抽取，模拟真实 DataLoader 的行为。

**代码说明**：
- 共训练 10 个 epoch，batch_size=32，学习率 0.01，使用 SGD 优化器。
- 每个 epoch 随机打乱 800 个训练样本，分批前向→交叉熵损失→反向→更新权重。
- 每 2 个 epoch 在 200 个测试样本上评估准确率。

**预期结果**：
- loss 从约 2.3 逐渐下降（模型在学习拟合训练数据）
- test_acc 约在 0.05~0.15 波动（因为使用随机数据，标签与图像无真实关联，10 类随机猜测期望为 0.10）
- `教师网络训练完成, 最终测试准确率: 约 0.10`

> **为什么准确率约 0.10？** 训练集和测试集标签都是随机生成的，图像与标签之间不存在真实映射关系。教师虽然能在训练集上过拟合，但无法泛化到测试集。10 类问题的随机猜测期望准确率为 1/10=0.10。在真实数据集（如 MNIST）上，教师准确率通常可达 0.95+。


In [ ]:
EPOCHS = 10
BATCH_SIZE = 32
LR = 0.01

optimizer_t = torch.optim.SGD(teacher.parameters(), lr=LR)
criterion = nn.CrossEntropyLoss()

teacher.train()
print('===== 开始训练教师网络 =====')

for epoch in range(EPOCHS):
    # 随机打乱训练集
    perm = torch.randperm(NUM_TRAIN)
    epoch_loss = 0.0
    n_batch = 0
    for i in range(0, NUM_TRAIN, BATCH_SIZE):
        idx = perm[i:i+BATCH_SIZE]
        x = train_images[idx]
        y = train_labels[idx]

        logits = teacher(x)
        loss = criterion(logits, y)

        optimizer_t.zero_grad()
        loss.backward()
        optimizer_t.step()

        epoch_loss += loss.item()
        n_batch += 1

    if (epoch + 1) % 2 == 0 or epoch == 0:
        acc = evaluate(teacher, test_images, test_labels)
        print(f'epoch: {epoch+1:2d}/{EPOCHS}  loss: {epoch_loss/n_batch:.4f}  test_acc: {acc:.4f}')

acc_teacher = evaluate(teacher, test_images, test_labels)
print(f'\n✅ 教师网络训练完成, 最终测试准确率: {acc_teacher:.4f}')

## 八、实现蒸馏损失函数

这是本实验的 **核心**。蒸馏损失由两部分组成：

### 8.1 硬标签损失（Hard Loss）

学生输出与 **真实标签** 的交叉熵——和普通训练一样：

$$L_{hard} = \text{CrossEntropy}(\text{student\_logits}, \text{labels})$$

### 8.2 软标签损失（Soft Loss）

学生软分布与 **教师软分布** 的 KL 散度：

$$L_{soft} = T^2 \cdot \text{KL}(\text{softmax}(z_s/T) \| \text{softmax}(z_t/T))$$

- `z_s`：学生 logits，`z_t`：教师 logits
- 乘 $T^2$ 是因为温度 softmax 会使梯度缩小 $1/T^2$，乘回来保持量级一致

### 8.3 总损失

$$L = \alpha \cdot L_{hard} + (1 - \alpha) \cdot L_{soft}$$

> 💡 `F.kl_div` 的输入约定：第一个参数是 **log概率**，第二个是 **概率**。
> 所以学生用 `log_softmax`，教师用 `softmax`。

In [ ]:
def distillation_loss(student_logits, teacher_logits, labels, T=4.0, alpha=0.5):
    """
    知识蒸馏损失函数
    
    参数:
        student_logits: 学生网络输出 logits
        teacher_logits: 教师网络输出 logits (已detach, 不反传给教师)
        labels:          真实标签
        T:               蒸馏温度 (越大分布越平滑)
        alpha:           硬损失权重 (0~1)
    返回:
        total_loss, hard_loss, soft_loss
    """
    # ---- 硬标签损失: 学生 vs 真实标签 ----
    hard_loss = F.cross_entropy(student_logits, labels)

    # ---- 软标签损失: 学生软分布 vs 教师软分布 (KL散度) ----
    soft_student = F.log_softmax(student_logits / T, dim=1)
    soft_teacher = F.softmax(teacher_logits / T, dim=1)
    soft_loss = F.kl_div(soft_student, soft_teacher, reduction='batchmean') * (T * T)

    # ---- 总损失: 加权组合 ----
    total_loss = alpha * hard_loss + (1.0 - alpha) * soft_loss
    return total_loss, hard_loss, soft_loss

# 测试一下损失函数
dummy_s = torch.randn(4, 10)
dummy_t = torch.randn(4, 10)
dummy_y = torch.randint(0, 10, (4,))
tl, hl, sl = distillation_loss(dummy_s, dummy_t, dummy_y, T=4.0, alpha=0.5)
print(f'✅ 蒸馏损失函数测试: total={tl.item():.4f}  hard={hl.item():.4f}  soft={sl.item():.4f}')

## 九、蒸馏训练学生网络（有教师指导）

现在用 **蒸馏损失** 训练学生网络。关键步骤：

1. 教师网络设为 `eval()` 模式，且 `no_grad()`——教师只做推理，不更新参数
2. 学生网络设为 `train()` 模式，正常前向 + 反向
3. 每个 batch：
   - 学生前向得到 `student_logits`
   - 教师前向得到 `teacher_logits`（**detach**，梯度不回传教师）
   - 计算 `distillation_loss`，反传更新学生

### 超参数说明

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">参数</th>
<th style="text-align: left;">值</th>
<th style="text-align: left;">说明</th>
</tr>
<tr>
<td style="text-align: left;">温度 T</td>
<td style="text-align: left;">4.0</td>
<td style="text-align: left;">软化教师输出，暴露暗知识</td>
</tr>
<tr>
<td style="text-align: left;">α</td>
<td style="text-align: left;">0.5</td>
<td style="text-align: left;">硬损失与软损失各占一半</td>
</tr>
<tr>
<td style="text-align: left;">epochs</td>
<td style="text-align: left;">10</td>
<td style="text-align: left;">训练轮数</td>
</tr>
<tr>
<td style="text-align: left;">lr</td>
<td style="text-align: left;">0.01</td>
<td style="text-align: left;">学习率</td>
</tr>
</table>

**超参数选择说明**：
- **温度 T=4.0**：经验表明 T=2~10 是大多数任务的最佳范围。T=4 能适度软化教师分布，既暴露了暗知识又保留了判别力。
- **α=0.5**：硬损失和软损失各占一半。α 过大会退化为普通训练（忽略教师知识），过小则可能忽视真实标签导致分类错误。
- **epochs=10, lr=0.01**：与教师训练保持一致，便于公平对比。

**预期结果**：
- loss（总损失）逐渐下降，其中 hard loss 和 soft loss 各自下降
- test_acc 约在 0.05~0.15（随机数据下，与教师类似）
- `蒸馏训练完成, 学生(蒸馏)测试准确率: 约 0.10`

> **在真实数据集上的预期**：如果使用 MNIST 真实数据，教师准确率约 0.98，学生(蒸馏)约 0.97，学生(基线)约 0.95，蒸馏收益约 +2%。本实验用随机数据，蒸馏收益不明显，但流程完整。


In [ ]:
TEMPERATURE = 4.0   # 蒸馏温度
ALPHA = 0.5          # 硬损失权重

student_distill = StudentNet().to(device)  # 蒸馏训练的学生
optimizer_sd = torch.optim.SGD(student_distill.parameters(), lr=LR)

teacher.eval()  # 教师设为评估模式
student_distill.train()
print('===== 开始蒸馏训练学生网络 =====')
print(f'(温度T={TEMPERATURE}, alpha={ALPHA})')

for epoch in range(EPOCHS):
    perm = torch.randperm(NUM_TRAIN)
    epoch_loss = 0.0
    epoch_hard = 0.0
    epoch_soft = 0.0
    n_batch = 0
    for i in range(0, NUM_TRAIN, BATCH_SIZE):
        idx = perm[i:i+BATCH_SIZE]
        x = train_images[idx]
        y = train_labels[idx]

        # 学生前向
        student_logits = student_distill(x)

        # 教师前向 (不计算梯度)
        with torch.no_grad():
            teacher_logits = teacher(x)

        # 蒸馏损失
        loss, hard_l, soft_l = distillation_loss(
            student_logits, teacher_logits, y,
            T=TEMPERATURE, alpha=ALPHA)

        optimizer_sd.zero_grad()
        loss.backward()
        optimizer_sd.step()

        epoch_loss += loss.item()
        epoch_hard += hard_l.item()
        epoch_soft += soft_l.item()
        n_batch += 1

    if (epoch + 1) % 2 == 0 or epoch == 0:
        acc = evaluate(student_distill, test_images, test_labels)
        print(f'epoch: {epoch+1:2d}/{EPOCHS}  '
              f'loss: {epoch_loss/n_batch:.4f}  '
              f'(hard:{epoch_hard/n_batch:.4f} soft:{epoch_soft/n_batch:.4f})  '
              f'test_acc: {acc:.4f}')

acc_distill = evaluate(student_distill, test_images, test_labels)
print(f'\n✅ 蒸馏训练完成, 学生(蒸馏)测试准确率: {acc_distill:.4f}')

## 十、普通训练学生网络（无蒸馏，作为基线）

为了证明蒸馏 **真的有效**，我们用完全相同的架构、相同的数据、相同的超参数，
但 **只用硬标签**（标准交叉熵）训练另一个学生网络作为对比基线。

```
学生(蒸馏) = 蒸馏损失训练  →  学了教师的暗知识
学生(基线) = 交叉熵训练    →  只学硬标签
```

> 如果蒸馏有效，学生(蒸馏) 的精度应 **高于** 学生(基线)。

In [ ]:
student_baseline = StudentNet().to(device)  # 无蒸馏训练的学生
optimizer_sb = torch.optim.SGD(student_baseline.parameters(), lr=LR)

student_baseline.train()
print('===== 开始普通训练学生网络(无蒸馏, 基线) =====')

for epoch in range(EPOCHS):
    perm = torch.randperm(NUM_TRAIN)
    epoch_loss = 0.0
    n_batch = 0
    for i in range(0, NUM_TRAIN, BATCH_SIZE):
        idx = perm[i:i+BATCH_SIZE]
        x = train_images[idx]
        y = train_labels[idx]

        logits = student_baseline(x)
        loss = criterion(logits, y)  # 只用硬标签交叉熵

        optimizer_sb.zero_grad()
        loss.backward()
        optimizer_sb.step()

        epoch_loss += loss.item()
        n_batch += 1

    if (epoch + 1) % 2 == 0 or epoch == 0:
        acc = evaluate(student_baseline, test_images, test_labels)
        print(f'epoch: {epoch+1:2d}/{EPOCHS}  loss: {epoch_loss/n_batch:.4f}  test_acc: {acc:.4f}')

acc_baseline = evaluate(student_baseline, test_images, test_labels)
print(f'\n✅ 普通训练完成, 学生(基线)测试准确率: {acc_baseline:.4f}')

## 十一、三者对比：教师 vs 学生(蒸馏) vs 学生(基线)

现在把三个模型放在一起对比：

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">模型</th>
<th style="text-align: left;">参数量</th>
<th style="text-align: left;">训练方式</th>
<th style="text-align: left;">期望精度</th>
</tr>
<tr>
<td style="text-align: left;">教师</td>
<td style="text-align: left;">多</td>
<td style="text-align: left;">标准CE</td>
<td style="text-align: left;">最高</td>
</tr>
<tr>
<td style="text-align: left;">学生(蒸馏)</td>
<td style="text-align: left;">少</td>
<td style="text-align: left;">蒸馏损失</td>
<td style="text-align: left;">接近教师</td>
</tr>
<tr>
<td style="text-align: left;">学生(基线)</td>
<td style="text-align: left;">少</td>
<td style="text-align: left;">标准CE</td>
<td style="text-align: left;">低于蒸馏</td>
</tr>
</table>

> 蒸馏的收益 = 学生(蒸馏)精度 - 学生(基线)精度。
> 在真实数据集上这个差距通常更明显。

**三者对比预期结果**：
- **准确率**：教师 ≈ 学生(蒸馏) ≥ 学生(基线)。在真实数据上，教师最高，学生(蒸馏)接近教师且高于学生(基线)。
- **参数量**：教师 >> 学生(蒸馏) = 学生(基线)。学生参数量仅为教师的约 6%。
- **推理时间**：教师 > 学生(蒸馏) ≈ 学生(基线)。学生推理更快（参数少、计算量小）。
- **模型大小**：教师 > 学生(蒸馏) ≈ 学生(基线)。学生体积更小。
- **蒸馏收益**：学生(蒸馏)准确率 - 学生(基线)准确率。在随机数据上可能为 0±0.02（无显著差异），在真实数据上通常为 +1~5%。


In [ ]:
time_teacher  = measure_time(teacher, device)
time_distill  = measure_time(student_distill, device)
time_baseline = measure_time(student_baseline, device)

size_teacher  = get_model_size(teacher)
size_distill  = get_model_size(student_distill)
size_baseline = get_model_size(student_baseline)

print('=' * 60)
print('              三者对比总结')
print('=' * 60)

print('\n【准确率对比】')
print(f'  教师(大模型):        {acc_teacher:.4f}')
print(f'  学生(蒸馏):          {acc_distill:.4f}  ← 学了教师的暗知识')
print(f'  学生(基线/无蒸馏):   {acc_baseline:.4f}  ← 只学硬标签')
print(f'  蒸馏收益:            {acc_distill - acc_baseline:+.4f}  (蒸馏比基线高多少)')

print('\n【参数量对比】')
print(f'  教师:    {teacher_params:>10,}')
print(f'  学生:    {student_params:>10,}  (仅为教师的 {100*student_params/teacher_params:.2f}%)')

print('\n【推理时间对比】(200次)')
print(f'  教师:          {time_teacher:.4f}s')
print(f'  学生(蒸馏):    {time_distill:.4f}s')
print(f'  学生(基线):    {time_baseline:.4f}s')

print('\n【模型大小对比】')
print(f'  教师:          {size_teacher:.2f} KB')
print(f'  学生(蒸馏):    {size_distill:.2f} KB')
print(f'  学生(基线):    {size_baseline:.2f} KB')

print('=' * 60)

## 十二、可视化：温度软化的效果

取一个测试样本，看教师 logits 在 **不同温度** 下的 softmax 分布，
直观理解温度如何 "暴露暗知识"。

- T=1：分布尖锐，几乎只有一类有概率
- T=4：分布平滑，能看到各类间的相对关系
- T=10：更平滑，暗知识更明显

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np

# 取一个样本
sample_x = test_images[0:1]
with torch.no_grad():
    teacher_logits = teacher(sample_x).cpu().numpy().flatten()

temperatures = [1, 2, 4, 10]
fig, axes = plt.subplots(1, len(temperatures), figsize=(16, 4))

for ax, T in zip(axes, temperatures):
    # 温度softmax
    scaled = teacher_logits / T
    probs = np.exp(scaled) / np.exp(scaled).sum()
    colors = ['coral' if i == probs.argmax() else 'steelblue' for i in range(10)]
    ax.bar(range(10), probs, color=colors, alpha=0.85)
    ax.set_title(f'Temperature T={T}')
    ax.set_xlabel('Class')
    ax.set_ylabel('Probability')
    ax.set_xticks(range(10))
    ax.set_ylim(0, 1.0)

plt.suptitle('Teacher Output Softmax at Different Temperatures\n(red = top class, blue = others = dark knowledge)',
             fontsize=12)
plt.tight_layout()
plt.show()

print('观察: T越大, 非top类的概率越高 → 暗知识(类间关系)越明显')
print('这就是学生网络要学习的额外信息!')

## 十三、可视化：学生输出分布对比

取同一样本，对比 **教师 / 学生(蒸馏) / 学生(基线)** 三者的输出概率分布，
看蒸馏学生是否更好地模仿了教师的分布形状。

In [ ]:
with torch.no_grad():
    t_logits = teacher(sample_x).cpu().numpy().flatten()
    d_logits = student_distill(sample_x).cpu().numpy().flatten()
    b_logits = student_baseline(sample_x).cpu().numpy().flatten()

def softmax(x):
    e = np.exp(x - x.max())
    return e / e.sum()

t_probs = softmax(t_logits)
d_probs = softmax(d_logits)
b_probs = softmax(b_logits)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

titles = ['Teacher', 'Student (Distill)', 'Student (Baseline)']
probs_list = [t_probs, d_probs, b_probs]
colors_list = ['steelblue', 'coral', 'seagreen']

for ax, title, probs, color in zip(axes, titles, probs_list, colors_list):
    ax.bar(range(10), probs, color=color, alpha=0.85)
    ax.set_title(title)
    ax.set_xlabel('Class')
    ax.set_ylabel('Probability')
    ax.set_xticks(range(10))
    ax.set_ylim(0, 1.0)

plt.suptitle('Output Distribution Comparison on Same Sample', fontsize=12)
plt.tight_layout()
plt.show()

# 计算分布相似度 (KL散度越小越相似)
kl_distill  = (softmax(d_logits) * (np.log(softmax(d_logits)+1e-9) - np.log(softmax(t_logits)+1e-9))).sum()
kl_baseline = (softmax(b_logits) * (np.log(softmax(b_logits)+1e-9) - np.log(softmax(t_logits)+1e-9))).sum()
print(f'与教师分布的KL散度:')
print(f'  学生(蒸馏):  {kl_distill:.4f}  ← 越小说明越像教师')
print(f'  学生(基线):  {kl_baseline:.4f}')

## 十四、蒸馏过程损失曲线分析

重新记录蒸馏训练过程中 **硬损失** 和 **软损失** 的变化曲线，
理解两个损失项在训练中是如何各自下降的。

In [ ]:
# 重新跑一次蒸馏训练并记录曲线
student_curve = StudentNet().to(device)
optimizer_sc = torch.optim.SGD(student_curve.parameters(), lr=LR)

teacher.eval()
student_curve.train()

hard_losses = []
soft_losses = []
total_losses = []
acc_curve = []

for epoch in range(EPOCHS):
    perm = torch.randperm(NUM_TRAIN)
    e_hard, e_soft, e_total = 0.0, 0.0, 0.0
    n_batch = 0
    for i in range(0, NUM_TRAIN, BATCH_SIZE):
        idx = perm[i:i+BATCH_SIZE]
        x = train_images[idx]
        y = train_labels[idx]
        s_logits = student_curve(x)
        with torch.no_grad():
            t_logits = teacher(x)
        loss, hl, sl = distillation_loss(s_logits, t_logits, y, T=TEMPERATURE, alpha=ALPHA)
        optimizer_sc.zero_grad()
        loss.backward()
        optimizer_sc.step()
        e_hard += hl.item(); e_soft += sl.item(); e_total += loss.item()
        n_batch += 1
    hard_losses.append(e_hard / n_batch)
    soft_losses.append(e_soft / n_batch)
    total_losses.append(e_total / n_batch)
    acc_curve.append(evaluate(student_curve, test_images, test_labels))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(range(1, EPOCHS+1), hard_losses, 'o-', label='Hard Loss (vs true label)', color='steelblue')
axes[0].plot(range(1, EPOCHS+1), soft_losses, 's-', label='Soft Loss (vs teacher)', color='coral')
axes[0].plot(range(1, EPOCHS+1), total_losses, '^-', label='Total Loss', color='seagreen')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Distillation Training Loss Curves')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(range(1, EPOCHS+1), acc_curve, 'o-', color='purple')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Test Accuracy')
axes[1].set_title('Student (Distill) Accuracy Curve')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print('观察: 硬损失和软损失同时下降, 学生在两者驱动下逐步逼近教师精度')

## 十五、探索：温度 T 对蒸馏效果的影响

温度 T 是蒸馏最关键的超参数。我们尝试不同温度，观察学生精度变化：

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">T</th>
<th style="text-align: left;">特点</th>
</tr>
<tr>
<td style="text-align: left;">1</td>
<td style="text-align: left;">几乎无软化，接近普通训练</td>
</tr>
<tr>
<td style="text-align: left;">2~4</td>
<td style="text-align: left;">常用范围，暗知识适中</td>
</tr>
<tr>
<td style="text-align: left;">8~10</td>
<td style="text-align: left;">高度软化，暗知识多但可能噪声也多</td>
</tr>
</table>

**温度对蒸馏效果的影响详解**：
- **T=1**：几乎无软化，教师输出分布尖锐（接近 one-hot），软标签携带的信息与硬标签几乎相同，蒸馏退化为普通训练，收益最小。
- **T=2~4**：适度软化，暗知识适中。教师输出暴露了"最可能是 X，其次像 Y"的类间关系，学生能从中学到有用的正则化信息。这是大多数任务的最佳范围。
- **T=8~10**：高度软化，分布接近均匀。暗知识很多，但也可能引入噪声——当所有类别概率接近相等时，教师输出几乎不提供判别信息。
- **最佳温度**因任务而异，一般规律是：类别数越多、教师越强，最佳 T 越大。

**预期结果**：
- 打印不同 T 值下学生(蒸馏)的准确率，以及基线和教师的准确率作为参考线
- 在真实数据上，T=2~4 通常表现最好；T=1 和 T=10 通常较差
- 在随机数据上，各温度的准确率差异可能不显著（约 0.10±0.02）


In [ ]:
temperatures_to_try = [1, 2, 4, 8, 10]
results = []

print('===== 不同温度蒸馏实验 =====')
for T in temperatures_to_try:
    s_model = StudentNet().to(device)
    opt = torch.optim.SGD(s_model.parameters(), lr=LR)
    teacher.eval()
    s_model.train()
    for epoch in range(EPOCHS):
        perm = torch.randperm(NUM_TRAIN)
        for i in range(0, NUM_TRAIN, BATCH_SIZE):
            idx = perm[i:i+BATCH_SIZE]
            x = train_images[idx]
            y = train_labels[idx]
            s_logits = s_model(x)
            with torch.no_grad():
                t_logits = teacher(x)
            loss, _, _ = distillation_loss(s_logits, t_logits, y, T=float(T), alpha=ALPHA)
            opt.zero_grad()
            loss.backward()
            opt.step()
    acc = evaluate(s_model, test_images, test_labels)
    results.append((T, acc))
    print(f'  T={T:2d}  →  学生(蒸馏)准确率: {acc:.4f}')

print(f'\n  基线(无蒸馏)准确率: {acc_baseline:.4f}')
print(f'  教师准确率:        {acc_teacher:.4f}')

# 画图
fig, ax = plt.subplots(figsize=(8, 5))
Ts = [r[0] for r in results]
As = [r[1] for r in results]
ax.plot(Ts, As, 'o-', color='coral', label='Student (Distill)')
ax.axhline(y=acc_baseline, color='steelblue', linestyle='--', label=f'Baseline (no distill) = {acc_baseline:.4f}')
ax.axhline(y=acc_teacher, color='seagreen', linestyle='--', label=f'Teacher = {acc_teacher:.4f}')
ax.set_xlabel('Temperature T')
ax.set_ylabel('Test Accuracy')
ax.set_title('Effect of Temperature on Distillation')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 十六、学习总结与思考

### ✅ 本实验你完成了

1. **理解** 知识蒸馏的原理：教师→学生的知识迁移
2. **搭建** 教师(大CNN) 和 学生(小CNN) 网络
3. **训练** 教师网络获得高精度
4. **实现** 蒸馏损失 = α·硬损失 + (1-α)·T²·软损失
5. **蒸馏训练** 学生网络，并对比 **无蒸馏基线**
6. **可视化** 温度软化效果、输出分布对比、损失曲线
7. **探索** 不同温度 T 对蒸馏效果的影响

### 🤔 课后思考

1. 把 α 改成 0.1 或 0.9，观察学生精度变化——**硬标签与软标签的权衡**
2. 为什么温度 T 太大或太小都不好？（提示：太小无暗知识，太大分布过于均匀失去判别力）
3. 如果教师和学生架构完全相同，蒸馏还有意义吗？（提示：自蒸馏 Self-Distillation）
4. 蒸馏和剪枝、量化有什么区别？三者如何组合使用？
5. 在昇腾 910 上，如何把蒸馏后的学生小模型部署到端侧获得加速？
6. 除了 KL 散度，还可以用什么度量学生与教师分布的差异？（提示：MSE、JS散度、Attention Transfer）

### 🔑 核心结论

> - **知识蒸馏**：让小模型学大模型的输出分布（含暗知识），获得比独立训练更高的精度
> - **温度软化**：T>1 使分布平滑，暴露类间关系；T² 补偿梯度缩放
> - **双损失加权**：硬标签保底 + 软标签迁移知识，α 控制两者比重
> - **蒸馏的价值**：推理用小模型（快、小），精度逼近大模型，适合昇腾等硬件部署

### 📚 延伸阅读

- 论文：Hinton et al. *Distilling the Knowledge in a Neural Network* (2015)
- 论文：*FitNets: Hints for Thin Deep Nets* (Romero et al., 2015) — 中间层蒸馏
- 论文：*Self-Distillation* — 自蒸馏，教师与学生同架构
- 昇腾开发文档：https://www.hiascend.com/document

---

> 🎉 恭喜完成知识蒸馏实验！请尝试修改温度 T、α、网络结构等参数重新运行，加深理解。

---

## 课后练习

请根据本实验内容完成以下题目进行自测。


**第1题**（单选题）知识蒸馏的核心思想是？

- A. 用大模型教小模型
- B. 降低数据精度
- C. 删除权重
- D. 增加层数


In [ ]:
q1 = ''  # 填入你的选项，如 'A'
print(f'第1题答案已记录：{q1}' if q1 else '请填入答案并运行本单元格')

**第2题**（单选题）教师网络和学生网络的关系是？

- A. 教师比学生小
- B. 教师比学生大，精度高
- C. 两者相同
- D. 学生比教师大


In [ ]:
q2 = ''  # 填入你的选项，如 'B'
print(f'第2题答案已记录：{q2}' if q2 else '请填入答案并运行本单元格')

**第3题**（单选题）温度软化（Temperature Softmax）中 T 的作用是？

- A. 提高精度
- B. 使分布平滑，暴露暗知识
- C. 加速训练
- D. 减少参数


In [ ]:
q3 = ''  # 填入你的选项，如 'B'
print(f'第3题答案已记录：{q3}' if q3 else '请填入答案并运行本单元格')

**第4题**（单选题）蒸馏损失函数由哪两部分组成？

- A. L1 损失和 L2 损失
- B. 硬标签损失和软标签损失
- C. 训练损失和验证损失
- D. 前向损失和反向损失


In [ ]:
q4 = ''  # 填入你的选项，如 'B'
print(f'第4题答案已记录：{q4}' if q4 else '请填入答案并运行本单元格')

**第5题**（单选题）软标签损失使用什么度量学生与教师分布的差异？

- A. MSE
- B. KL 散度
- C. 交叉熵
- D. 余弦相似度


In [ ]:
q5 = ''  # 填入你的选项，如 'A'
print(f'第5题答案已记录：{q5}' if q5 else '请填入答案并运行本单元格')

**第6题**（单选题）蒸馏损失中乘以 T² 的原因是？

- A. 提高精度
- B. 补偿温度对梯度的缩放
- C. 加速收敛
- D. 减少参数


In [ ]:
q6 = ''  # 填入你的选项，如 'B'
print(f'第6题答案已记录：{q6}' if q6 else '请填入答案并运行本单元格')

**第7题**（单选题）暗知识（Dark Knowledge）是指？

- A. 模型权重
- B. 教师输出中非 top 类的概率分布
- C. 训练数据
- D. 模型结构


In [ ]:
q7 = ''  # 填入你的选项，如 'B'
print(f'第7题答案已记录：{q7}' if q7 else '请填入答案并运行本单元格')

**第8题**（单选题）蒸馏训练时教师网络需要更新参数吗？

- A. 需要
- B. 不需要，只做推理（eval + no_grad）
- C. 偶尔需要
- D. 取决于温度


In [ ]:
q8 = ''  # 填入你的选项，如 'A'
print(f'第8题答案已记录：{q8}' if q8 else '请填入答案并运行本单元格')

**第9题**（单选题）温度 T 太大或太小的问题分别是？

- A. 太小无暗知识，太大分布过于均匀失去判别力
- B. 太小太快，太慢太慢
- C. 太小欠拟合，太大过拟合
- D. 没有问题


In [ ]:
q9 = ''  # 填入你的选项，如 'B'
print(f'第9题答案已记录：{q9}' if q9 else '请填入答案并运行本单元格')

**第10题**（单选题）知识蒸馏的最终价值是？

- A. 增加模型大小
- B. 推理用小模型（快、小），精度逼近大模型
- C. 提高训练速度
- D. 减少数据需求


In [ ]:
q10 = ''  # 填入你的选项，如 'B'
print(f'第10题答案已记录：{q10}' if q10 else '请填入答案并运行本单元格')

**全部作答完成后，运行下方代码查看批改结果：**


In [ ]:
import sys
from pathlib import Path

for candidate in (Path.cwd() / 'answer', Path.cwd().parent / 'answer'):
    if candidate.exists():
        sys.path.insert(0, str(candidate.resolve()))
        break
else:
    raise FileNotFoundError('Cannot find answer directory')
from grade_06 import grade
grade(globals())

## 参考资料

- 论文：Hinton et al. *Distilling the Knowledge in a Neural Network* (2015)
- [昇腾社区文档](https://hiascend.com/document)
